# Extract Central Banks And Governors From Wikipedia

Flujo completo desde Wikipedia hasta una tabla limpia por persona.

## Qué hace

1. Entra a `List of central banks` en Wikipedia y extrae país, nombre del banco y URL de Wikipedia de cada banco central (columna 2 de la tabla, no la moneda).
2. Entra a la página Wikipedia de cada banco central.
3. Busca tablas (`wikitable`) cuyo contexto contenga `governor`, `president`, `chair` o `key people`, y filas de infobox con los mismos patrones.
4. Guarda todas las filas sin perder información en `central_bank_governors_df`.
5. Explota `row_data` a formato `key/value` largo en `central_bank_governors_items_df`.
6. Detecta filas header con claves numéricas (`1`, `2`, `4`...) y las usa para renombrar las filas de datos del mismo grupo.
7. Extrae el nombre limpio, año de inicio y año de fin de cada persona, manejando múltiples formatos de tabla y casos especiales (múltiples personas en un campo, cargos pegados al nombre, años de vida, etc.).
8. Genera `central_bank_governors_long_df` con una fila por persona.

## Outputs

### `central_banks.csv`
Lista de bancos centrales con su URL de Wikipedia.
- `country` · `central_bank` · `wikipedia_bank_url`

### `central_bank_governors.csv`
Una fila por fila de tabla extraída de Wikipedia. Contiene `row_data` como dict crudo.
- `country` · `central_bank` · `wikipedia_bank_url` · `source_type` · `source_label` · `row_data`

### `central_bank_governors_items.csv`
Versión explodida de `row_data`: una fila por cada elemento key/value.
- `country` · `central_bank` · `wikipedia_bank_url` · `source_type` · `source_label` · `row_key` · `row_value` · `row_data`

### `central_bank_governors_long.csv`
Una fila por persona detectada. Es el output principal.
- `country` · `central_bank` · `wikipedia_bank_url` · `source_type` · `source_label` · `wiki_name` · `cargo` · `start_year` · `end_year`

### `central_bank_governors_request_errors.csv`
Bancos donde falló la request HTTP.

## Separador
Todos los archivos usan `;` como separador para evitar conflictos con comas en nombres de países (`Korea, Democratic People's Republic of`) y nombres de personas.

In [ ]:
import ast
import json
import re
import os
import time
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; CentralBankNotebook/1.0)"}
WIKI_BANKS_URL = "https://en.wikipedia.org/wiki/List_of_central_banks"
ROLE_PATTERNS = ["governor", "governors", "president", "presidents", "chair", "chairs", "key people"]

NAME_KEYS = [
    "Name", "Governor", "President", "Chairman", "Chair",
    "Current governor", "Key people", "Name | Name",
    "Governor | Governor", "Governor | Governor.1",
    "Name | Governor of CBC (Guangzhou)",
    "Name (governor) | Name (governor)", "Name (Signature)",
    "Governor and Chairman", "President of the Bank of Guatemala",
    "Governor & Other Positions", "Board of Directors", "1",
]
START_KEYS = [
    "Took office", "Term of office | Start of term", "Term start",
    "Entered office", "Start", "From",
    "Term of office | Took office", "tenure start", "2",
]
END_KEYS = [
    "Left office", "Term of office | End of term", "Term end",
    "Exited office", "End", "Until", "Term expires",
    "Term of office | Left office", "tenure end", "4",
]
PERIOD_KEYS = ["Term", "Period", "Tenure", "In office", "Term of office",
               "Tenure length | Tenure length"]
HEADER_VALUES = {
    "name", "governor", "president", "chairman", "chair",
    "key people", "directors", "board of directors",
    "governors of the commonwealth bank of australia",
    "governors of the reserve bank of australia",
}
INVALID_NAME_RE = [
    r"^\d+$", r"^\(\d+\)$", r"^disestablished",
    r"^acting director", r"^director of", r"^deputy director",
    r"^executive assistant", r"^first deputy governor$", r"^second deputy governor$",
]


# ── Helpers ───────────────────────────────────────────────────────────────────

def clean_text(value):
    value = str(value or "")
    value = re.sub(r"\[[^\]]*\]", "", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip(" -–—,;")


def flatten_columns(columns):
    if not isinstance(columns, pd.MultiIndex):
        return [clean_text(col) for col in columns]
    flattened = []
    for col in columns:
        parts = [clean_text(x) for x in col if clean_text(x) and str(x) != "nan"]
        flattened.append(" | ".join(parts))
    return flattened


def table_context_text(table):
    pieces = []
    caption = table.find("caption")
    if caption:
        pieces.append(caption.get_text(" ", strip=True))
    rows = table.select("tr")
    if rows:
        pieces.append(" ".join(cell.get_text(" ", strip=True) for cell in rows[0].find_all(["th", "td"])))
    prev = table.find_previous(["h2", "h3", "h4"])
    if prev:
        pieces.append(prev.get_text(" ", strip=True))
    return clean_text(" | ".join(piece for piece in pieces if piece))


def sanitize_table_html(table):
    html = str(table)
    html = re.sub(r'colspan="(\d+);"', r'colspan="\1"', html)
    html = re.sub(r'rowspan="(\d+);"', r'rowspan="\1"', html)
    return html


def parse_html_table(table):
    return pd.read_html(StringIO(sanitize_table_html(table)))[0]


def parse_row_data(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return {}
    if isinstance(raw, dict):
        data = raw
    else:
        try:
            data = ast.literal_eval(raw)
        except Exception:
            return {}
    if not isinstance(data, dict):
        return {}
    return {str(k): clean_text(v) for k, v in data.items() if clean_text(v) and clean_text(v).lower() != "nan"}


def get_year(s):
    if not s:
        return ""
    if re.search(r"[Ii]ncumbent|[Pp]resent|[Cc]urrent", s):
        return "Incumbent"
    years = re.findall(r"\d{4}", s)
    return years[0] if years else s.strip()


def clean_name(name):
    name = re.sub(r"\s*,\s*(Governor|President|Chair|Director|Acting|Deputy|Vice).*", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*\(\d{4}\s*[–\-]\s*\d{4}\)", "", name)
    name = re.sub(r"\s*\(\d{4}[–\-]?\)", "", name)
    name = re.sub(r"\s*\(\d{4}-present\)", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*\(effective[^)]*\)", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*\(born\s+\d{4}\).*$", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*\([^)]*(?:governor|president|chair|director|economist|vice|acting)[^)]*\).*$", "", name, flags=re.IGNORECASE)
    name = name.replace(",", "")
    name = re.sub(r"\s+", " ", name)
    return name.strip()


def is_invalid(name):
    for pat in INVALID_NAME_RE:
        if re.match(pat, name, re.IGNORECASE):
            return True
    return False


def is_too_long(name):
    return len(name.split()) > 8


def looks_like_header_row(d):
    values = {clean_text(v).lower() for v in d.values() if clean_text(v)}
    return len(values & HEADER_VALUES) >= 2


def normalize_header_map(d):
    return {clean_text(k): clean_text(v) for k, v in d.items() if clean_text(k) and clean_text(v)}


def relabel_numeric_dict(d, header_map):
    relabeled = {}
    for k, v in d.items():
        kk = clean_text(k)
        vv = clean_text(v)
        if not vv:
            continue
        relabeled[header_map.get(kk, kk)] = vv
    return relabeled


def infer_cargo(source_label, name_key):
    text = f"{source_label} | {name_key}".lower()
    if "chair" in text:
        return "Chair"
    if "president" in text:
        return "President"
    if "governor" in text:
        return "Governor"
    if "key people" in text or "board" in text:
        return "Board Member"
    return ""


def split_multi_person(value):
    pattern = r"([A-ZÀ-ÿ][^(]{2,40?}?)\s*\(([^)]+)\)"
    matches = re.findall(pattern, value)
    if len(matches) > 1:
        return [(m[0].strip(), m[1].strip()) for m in matches]
    return None


# ── Scraping ──────────────────────────────────────────────────────────────────

def extract_table_rows(parsed_table, bank_row, context_text):
    parsed_table = parsed_table.copy()
    parsed_table.columns = flatten_columns(parsed_table.columns)
    records = []
    for _, row in parsed_table.iterrows():
        row_data = {
            clean_text(col): clean_text(row.get(col, ""))
            for col in parsed_table.columns
            if clean_text(row.get(col, ""))
        }
        if not row_data:
            continue
        records.append({
            "country": clean_text(bank_row.country),
            "central_bank": clean_text(bank_row.central_bank),
            "wikipedia_bank_url": clean_text(bank_row.wikipedia_bank_url),
            "source_type": "wikitable",
            "source_label": context_text,
            "row_data": row_data,
        })
    return records


def extract_infobox_rows(bank_row, soup):
    records = []
    infobox = soup.select_one("table.infobox")
    if not infobox:
        return records
    for tr in infobox.select("tr"):
        th = tr.find("th")
        td = tr.find("td")
        if not th or not td:
            continue
        label = clean_text(th.get_text(" ", strip=True))
        value = clean_text(td.get_text(" ", strip=True))
        if not label or not value:
            continue
        if any(pattern in label.lower() for pattern in ROLE_PATTERNS):
            records.append({
                "country": clean_text(bank_row.country),
                "central_bank": clean_text(bank_row.central_bank),
                "wikipedia_bank_url": clean_text(bank_row.wikipedia_bank_url),
                "source_type": "infobox",
                "source_label": label,
                "row_data": {label: value},
            })
    return records


def extract_governor_rows(bank_row):
    url = bank_row.wikipedia_bank_url
    if not url:
        return []
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
    except requests.RequestException as exc:
        return [{
            "country": clean_text(bank_row.country),
            "central_bank": clean_text(bank_row.central_bank),
            "wikipedia_bank_url": clean_text(url),
            "source_type": "request_error",
            "source_label": type(exc).__name__,
            "row_data": {"error": str(exc)},
        }]
    soup = BeautifulSoup(response.text, "html.parser")
    records = []
    for table in soup.select("table.wikitable"):
        context_text = table_context_text(table)
        if not any(pattern in context_text.lower() for pattern in ROLE_PATTERNS):
            continue
        try:
            parsed_table = parse_html_table(table)
        except ValueError:
            continue
        records.extend(extract_table_rows(parsed_table, bank_row, context_text))
    if records:
        return records
    return extract_infobox_rows(bank_row, soup)


# ── Name extraction (same logic as expand_row_data.py) ───────────────────────

def extract_person_rows(row, header_map):
    """
    Given a raw row and a header_map for numeric keys,
    returns a list of person dicts with name/start/end.
    """
    d = parse_row_data(row["row_data"])
    if not d:
        return []

    if looks_like_header_row(d):
        return []

    if header_map:
        d = relabel_numeric_dict(d, header_map)

    raw_name = next((clean_text(d[k]) for k in NAME_KEYS if clean_text(d.get(k, ""))), "")
    if not raw_name:
        return []

    start = next((get_year(clean_text(d[k])) for k in START_KEYS if clean_text(d.get(k, ""))), "")
    if not start:
        for k in PERIOD_KEYS:
            if clean_text(d.get(k, "")):
                yrs = re.findall(r"\d{4}", clean_text(d[k]))
                if yrs:
                    start = yrs[0]
                    break

    end = next((get_year(clean_text(d[k])) for k in END_KEYS if clean_text(d.get(k, ""))), "")
    if not end:
        for k in PERIOD_KEYS:
            if clean_text(d.get(k, "")):
                val = clean_text(d[k])
                if re.search(r"[Pp]resent|[Cc]urrent|[Ii]ncumbent", val):
                    end = "Incumbent"
                    break
                yrs = re.findall(r"\d{4}", val)
                if len(yrs) > 1:
                    end = yrs[-1]
                    break

    name_source = next((k for k in NAME_KEYS if clean_text(d.get(k, "")) == raw_name), "")
    cargo = infer_cargo(row["source_label"], name_source)

    # Handle multiple people in one field
    multi = split_multi_person(raw_name)
    if multi:
        results = []
        for name, role in multi:
            name = clean_name(name)
            if not name or name.lower() in HEADER_VALUES or is_invalid(name) or is_too_long(name):
                continue
            results.append({
                "country": row["country"],
                "central_bank": row["central_bank"],
                "wikipedia_bank_url": row["wikipedia_bank_url"],
                "source_type": row["source_type"],
                "source_label": row["source_label"],
                "wiki_name": name,
                "cargo": role,
                "start_year": start,
                "end_year": end,
            })
        return results

    name = clean_name(raw_name)
    if not name or name.lower() in HEADER_VALUES or is_invalid(name) or is_too_long(name):
        return []

    return [{
        "country": row["country"],
        "central_bank": row["central_bank"],
        "wikipedia_bank_url": row["wikipedia_bank_url"],
        "source_type": row["source_type"],
        "source_label": row["source_label"],
        "wiki_name": name,
        "cargo": cargo,
        "start_year": start,
        "end_year": end,
    }]


# ── Main pipeline ─────────────────────────────────────────────────────────────

# 1. Get central banks list with correct URLs (column index 2 = Central bank)
response = requests.get(WIKI_BANKS_URL, headers=HEADERS, timeout=30)
response.raise_for_status()

wiki_tables = pd.read_html(StringIO(response.text))
soup = BeautifulSoup(response.text, "html.parser")
html_tables = soup.select("table.wikitable")

if len(wiki_tables) < 2 or len(html_tables) < 2:
    raise ValueError("Expected at least two tables on the Wikipedia page")

alphabetical_html_table = html_tables[0]
alphabetical_records = []

for row in alphabetical_html_table.select("tr")[1:]:
    cols = row.find_all(["th", "td"])
    if len(cols) < 3:
        continue
    # Column 0 = Country, Column 1 = Currency, Column 2 = Central bank
    country = cols[0].get_text(" ", strip=True)
    bank_col = cols[2]
    central_bank = bank_col.get_text(" ", strip=True)
    link = bank_col.find("a", href=True)
    wikipedia_bank_url = ""
    if link and link["href"].startswith("/wiki/"):
        wikipedia_bank_url = "https://en.wikipedia.org" + link["href"]
    alphabetical_records.append({
        "country": country,
        "central_bank": central_bank,
        "wikipedia_bank_url": wikipedia_bank_url,
    })

alphabetical_banks_df = pd.DataFrame(alphabetical_records)
major_table_raw = wiki_tables[1].copy()
major_banks_df = major_table_raw.rename(
    columns={major_table_raw.columns[0]: "country", major_table_raw.columns[1]: "central_bank"}
)[["country", "central_bank"]].copy()
major_banks_df["wikipedia_bank_url"] = ""

for frame in (alphabetical_banks_df, major_banks_df):
    frame["country"] = frame["country"].astype(str).str.strip()
    frame["central_bank"] = frame["central_bank"].astype(str).str.strip()
    frame["wikipedia_bank_url"] = frame["wikipedia_bank_url"].astype(str).str.strip()

central_banks_df = pd.concat([alphabetical_banks_df, major_banks_df], ignore_index=True)
central_banks_df = central_banks_df.drop_duplicates(subset=["country", "central_bank"]).reset_index(drop=True)

print("central_banks_df", central_banks_df.shape)
display(central_banks_df.head(10))

# 2. Scrape all governor rows
governor_rows = []
for i, bank_row in enumerate(central_banks_df.itertuples(index=False), start=1):
    if i % 25 == 0:
        print(f"Processed {i}/{len(central_banks_df)} banks")
    governor_rows.extend(extract_governor_rows(bank_row))
    time.sleep(0.3)

central_bank_governors_df = pd.DataFrame(governor_rows)
request_errors_df = central_bank_governors_df[
    central_bank_governors_df["source_type"] == "request_error"
].copy()

print("central_bank_governors_df", central_bank_governors_df.shape)
print("request_errors_df", request_errors_df.shape)
display(central_bank_governors_df.head(20))

# 3. Build long format — one row per person
group_cols = ["country", "central_bank", "wikipedia_bank_url", "source_type", "source_label"]
long_rows = []

for _, group in central_bank_governors_df.groupby(group_cols, dropna=False, sort=False):
    # Detect header row to build key map for numeric-key tables
    header_map = {}
    for _, row in group.iterrows():
        d = parse_row_data(row["row_data"])
        if d and looks_like_header_row(d):
            header_map = normalize_header_map(d)
            break

    for _, row in group.iterrows():
        long_rows.extend(extract_person_rows(row, header_map))

central_bank_governors_long_df = pd.DataFrame(long_rows)

# Deduplicate
if not central_bank_governors_long_df.empty:
    central_bank_governors_long_df = central_bank_governors_long_df.drop_duplicates(
        subset=["country", "central_bank", "wiki_name", "start_year", "end_year"]
    ).reset_index(drop=True)

print("central_bank_governors_long_df", central_bank_governors_long_df.shape)
display(central_bank_governors_long_df.head(20))

# 4. Export with semicolon separator
os.makedirs("data", exist_ok=True)
central_banks_df.to_csv("/Users/sbc/projects/central-banks-board/CM/data/central_banks.csv", index=False, sep=";")
central_bank_governors_df.to_csv("/Users/sbc/projects/central-banks-board/CM/data/central_bank_governors.csv", index=False, sep=";")
central_bank_governors_long_df.to_csv("/Users/sbc/projects/central-banks-board/CM/data/central_bank_governors_long.csv", index=False, sep=";")
request_errors_df.to_csv("/Users/sbc/projects/central-banks-board/CM/data/central_bank_governors_request_errors.csv", index=False, sep=";")

print("Exported OK")


central_banks_df (216, 3)


,country,central_bank,wikipedia_bank_url
0,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...
1,Afghanistan,Da Afghanistan Bank,https://en.wikipedia.org/wiki/Da_Afghanistan_Bank
2,Albania,Bank of Albania,https://en.wikipedia.org/wiki/Bank_of_Albania
3,Algeria,Bank of Algeria,https://en.wikipedia.org/wiki/Bank_of_Algeria
4,Andorra,No central bank; uses the Euro as its domestic...,https://en.wikipedia.org/wiki/Euro
5,Angola,National Bank of Angola,https://en.wikipedia.org/wiki/National_Bank_of...
6,Anguilla,Eastern Caribbean Central Bank,https://en.wikipedia.org/wiki/Eastern_Caribbea...
7,Antigua and Barbuda,Eastern Caribbean Central Bank,https://en.wikipedia.org/wiki/Eastern_Caribbea...
8,Argentina,Central Bank of Argentina,https://en.wikipedia.org/wiki/Central_Bank_of_...
9,Armenia,Central Bank of Armenia,https://en.wikipedia.org/wiki/Central_Bank_of_...


Processed 25/216 banks
Processed 50/216 banks
Processed 75/216 banks
Processed 100/216 banks
Processed 125/216 banks
Processed 150/216 banks
Processed 175/216 banks
Processed 200/216 banks
central_bank_governors_df (860, 6)
request_errors_df (0, 6)


,country,central_bank,wikipedia_bank_url,source_type,source_label,row_data
0,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,"{'': '#', '1': 'Name', '2': 'From', '3': 'nan'..."
1,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,"{'': 'nan', '1': 'Daur Barganjia', '2': '1996'..."
2,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,"{'': 'nan', '1': 'Boris Zhirov', '2': 'nan', '..."
3,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,"{'': 'nan', '1': 'Emma Tania', '2': '10 Novemb..."
4,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,"{'': 'nan', '1': 'Illarion Argun', '2': '10 Ju..."
5,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,"{'': 'nan', '1': 'Illarion Argun', '2': '29 Ma..."
6,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,"{'': 'nan', '1': 'Illarion Argun', '2': '1 Jun..."
7,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,"{'': 'nan', '1': 'Beslan Baratelia', '2': '29 ..."
8,Afghanistan,Da Afghanistan Bank,https://en.wikipedia.org/wiki/Da_Afghanistan_Bank,infobox,Governor,{'Governor': 'Noor Ahmad Agha'}
9,Albania,Bank of Albania,https://en.wikipedia.org/wiki/Bank_of_Albania,infobox,Governor,{'Governor': 'Gent Sejko'}


central_bank_governors_long_df (809, 9)


,country,central_bank,wikipedia_bank_url,source_type,source_label,wiki_name,cargo,start_year,end_year
0,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,Daur Barganjia,Chair,1996,1998
1,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,Boris Zhirov,Chair,,2004
2,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,Emma Tania,Chair,2004,2005
3,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,Illarion Argun,Chair,2005,2011
4,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,Illarion Argun,Chair,2011,2014
5,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,Illarion Argun,Chair,2014,2014
6,Abkhazia *,National Bank of the Republic of Abkhazia,https://en.wikipedia.org/wiki/National_Bank_of...,wikitable,# Name From Until President Comments | Chairma...,Beslan Baratelia,Chair,2014,Incumbent
7,Afghanistan,Da Afghanistan Bank,https://en.wikipedia.org/wiki/Da_Afghanistan_Bank,infobox,Governor,Noor Ahmad Agha,Governor,,
8,Albania,Bank of Albania,https://en.wikipedia.org/wiki/Bank_of_Albania,infobox,Governor,Gent Sejko,Governor,,
9,Algeria,Bank of Algeria,https://en.wikipedia.org/wiki/Bank_of_Algeria,infobox,Governor,Mouatassem Boudiaf,Governor,,


Exported OK
